In [1]:
import pandas as pd 
import numpy as np 


In [2]:
df = pd.read_csv('Heart-Disease.csv',sep=';')

In [3]:
df.head()

,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
0,0,18393,2,168,62.0,110,80,1,1,0,0,1,0
1,1,20228,1,156,85.0,140,90,3,1,0,0,1,1
2,2,18857,1,165,64.0,130,70,3,1,0,0,0,1
3,3,17623,2,169,82.0,150,100,1,1,0,0,1,1
4,4,17474,1,156,56.0,100,60,1,1,0,0,0,0


In [4]:
df['age'] = df['age'] / 365

In [5]:
df.isnull().sum()
df.head()

,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
0,0,50.391781,2,168,62.0,110,80,1,1,0,0,1,0
1,1,55.419178,1,156,85.0,140,90,3,1,0,0,1,1
2,2,51.663014,1,165,64.0,130,70,3,1,0,0,0,1
3,3,48.282192,2,169,82.0,150,100,1,1,0,0,1,1
4,4,47.873973,1,156,56.0,100,60,1,1,0,0,0,0


In [6]:
df.info()
df = df.drop('id',axis=1)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70000 entries, 0 to 69999
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id           70000 non-null  int64  
 1   age          70000 non-null  float64
 2   gender       70000 non-null  int64  
 3   height       70000 non-null  int64  
 4   weight       70000 non-null  float64
 5   ap_hi        70000 non-null  int64  
 6   ap_lo        70000 non-null  int64  
 7   cholesterol  70000 non-null  int64  
 8   gluc         70000 non-null  int64  
 9   smoke        70000 non-null  int64  
 10  alco         70000 non-null  int64  
 11  active       70000 non-null  int64  
 12  cardio       70000 non-null  int64  
dtypes: float64(2), int64(11)
memory usage: 6.9 MB


In [7]:
X = df.drop('cardio',axis=1)
y = df['cardio']

In [8]:
y.value_counts()

cardio
0    35021
1    34979
Name: count, dtype: int64

In [9]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42,stratify=y)

In [10]:
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

pipline = Pipeline([
    ('scaler',StandardScaler()),
    ('smote',SMOTE(random_state=42)),
    ('model', RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight='balanced'
    ))
])

In [11]:
pipline.fit(X_train,y_train)

,steps,"[('scaler', ...), ('smote', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,sampling_strategy,'auto'
,random_state,42
,k_neighbors,5
,n_estimators,100


In [12]:
y_prid = pipline.predict(X_test)

In [13]:
from sklearn.metrics import accuracy_score, classification_report

accuracy = accuracy_score(y_test, y_prid)

In [14]:
accuracy

0.713939393939394

In [15]:
print(classification_report(y_test,y_prid))

              precision    recall  f1-score   support

           0       0.71      0.73      0.72     11557
           1       0.72      0.70      0.71     11543

    accuracy                           0.71     23100
   macro avg       0.71      0.71      0.71     23100
weighted avg       0.71      0.71      0.71     23100



In [16]:
from sklearn.model_selection import StratifiedKFold, cross_validate
import numpy as np

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

results = cross_validate(
    pipline,
    X,
    y,
    cv=skf,
    scoring=['f1', 'accuracy', 'recall', 'precision']
)

print("F1 Score:", results['test_f1'])
print("Average F1:", np.mean(results['test_f1']))

print("Accuracy:", results['test_accuracy'])
print("Average Accuracy:", np.mean(results['test_accuracy']))

print("Recall:", results['test_recall'])
print("Average Recall:", np.mean(results['test_recall']))

print("Precision:", results['test_precision'])
print("Average Precision:", np.mean(results['test_precision']))

F1 Score: [0.71183496 0.71082673 0.70925365 0.71351038 0.70674508]
Average F1: 0.7104341602154507
Accuracy: [0.71564286 0.71592857 0.71228571 0.71721429 0.7115    ]
Average Accuracy: 0.7145142857142857
Recall: [0.70293066 0.69868496 0.70225843 0.70468839 0.69568325]
Average Recall: 0.7008491403849273
Precision: [0.72096774 0.72339796 0.71638962 0.72255606 0.71816438]
Average Precision: 0.7202951514952771


In [17]:
import joblib 
joblib.dump(pipline,'heart_model.pkl')
print('model saved successfuly')

model saved successfuly
